# Champs facultatifs vides — leboncoin-private

Constat sur le formulaire `/prix` : laisser vide l'accordéon « Informations complémentaires »
(puissance DIN, puissance fiscale, Crit'Air, portes, places, contrôle technique, couleur)
produit une estimation nettement plus basse. Ce carnet **mesure** le phénomène avant tout
remède (plan `docs/plans/2026-08-10-mesure-champs-manquants.md`).

**Hypothèse à vérifier.** Un champ vide devient une valeur manquante (NaN), et
`HistGradientBoostingRegressor` route les manquants selon ce qu'il a appris : dans les
annonces scrapées, « non renseigné » corrèle avec « annonce bâclée de véhicule pas cher ».
Au formulaire, « vide » veut dire « le vendeur ne sait pas ». Même encodage, deux
significations — un décalage entraînement/service porté par le *motif de manquance*
(MNAR — Missing Not At Random : la probabilité qu'une valeur manque dépend du type
d'annonce, pas du hasard).

Trois expériences, **sans aucun ré-entraînement** — on interroge le modèle réellement servi
(`app/models/prix.joblib` + `prix.json`) :

1. **A** — chaque annonce du jeu de test prédite deux fois : complète, puis accordéon masqué ;
2. **B** — l'hypothèse MNAR vérifiée dans les données (prix médian avec vs sans champ) ;
3. **C** — le cas utilisateur rejoué via `app/src/prix.estimer()` (la Clio de référence).


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Le carnet parle aux deux mondes : `ml/src` pour le nettoyage, `app/src` pour le contrat
# d'encodage partagé et le service — exactement les imports de `ml/src/entrainement.py`.
sys.path.insert(0, str(Path("../../src").resolve()))
sys.path.insert(0, str(Path("../../../app/src").resolve()))

from leboncoin import clean_leboncoin
from preparation import ETATS_GROUPES, FREQUENCE_INCONNUE, TRANCHES, construire_jeu

PARQUET = Path("../../data/leboncoin-private/raw/annonces.parquet")
PREMIUM = Path("../../references/premium_brand.csv")
MODELES = Path("../../../app/models")

import joblib, json
# SECURITE — joblib deroule du pickle (execution de code). Acceptable ici et seulement ici :
# l'artefact est produit par `ml/src/entrainement.py` dans ce depot et versionne avec le
# code — jamais televerse ni telecharge (meme justification que dans `app/src/prix.py`).
modeles = joblib.load(MODELES / "prix.joblib")          # {"q10", "q50", "q90"}
meta = json.loads((MODELES / "prix.json").read_text(encoding="utf-8"))
Q = meta["conformal_q"]
print(f"artefacts charges — conformal_q = {Q:+.2f} EUR, "
      f"couverture test enregistree = {meta['couverture_test']:.2%}")


artefacts charges — conformal_q = +145.35 EUR, couverture test enregistree = 79.48%


## 1. Reproduire le jeu de test — au bit près

Les chiffres « accordéon vide » ne valent que si le mode « complet » retrouve **exactement**
les métriques enregistrées dans `prix.json` (MAE 1 448 €, couverture 79,48 %). On rejoue donc
les dérivations et le découpage de `ml/src/entrainement.py` : âge depuis l'année seule, état
regroupé, split 60/20/20 avec `random_state=42`, fréquence du modèle lue dans la table de
l'artefact (celle comptée sur le jeu d'ajustement).


In [2]:
df, _ = clean_leboncoin(PARQUET, PREMIUM)
date_reference = pd.Timestamp(df["scraped_at"].max()).tz_localize(None)
df["age"] = (date_reference - pd.to_datetime(dict(year=df["annee"], month=1, day=1))).dt.days / 365.25
df["etat"] = df["etat"].map(ETATS_GROUPES)

from sklearn.model_selection import train_test_split
df_train, df_test = train_test_split(df, test_size=0.20, random_state=42)

# La table de frequences de l'ARTEFACT (comptee sur le jeu d'ajustement seul) : c'est elle
# que le service applique, donc elle que la mesure doit appliquer.
freq = meta["frequence_modele"]
df_test = df_test.copy()
df_test["modele_freq"] = df_test["modele"].map(freq).fillna(FREQUENCE_INCONNUE).astype("float64")

y_test = df_test["prix_eur"]
print(f"test : {len(df_test)} annonces (attendu {meta['metriques']['n_test']})")


test : 3977 annonces (attendu 3977)


In [3]:
from sklearn.metrics import mean_absolute_error

def fourchette(df_lignes: pd.DataFrame) -> pd.DataFrame:
    """Rejoue exactement la logique servie par `app/src/prix.py` : bornes conformalisees,
    remise en ordre, central rabattu dans l'intervalle."""
    X = construire_jeu(df_lignes, categories=meta["categories"])
    lo = modeles["q10"].predict(X) - Q
    hi = modeles["q90"].predict(X) + Q
    central = modeles["q50"].predict(X)
    lo, hi = np.minimum(lo, hi), np.maximum(lo, hi)
    central = np.clip(central, lo, hi)
    return pd.DataFrame({"bas": lo, "central": central, "haut": hi,
                         "largeur": hi - lo}, index=df_lignes.index)

complet = fourchette(df_test)

mae = mean_absolute_error(y_test, complet["central"])
couverture = ((y_test >= complet["bas"]) & (y_test <= complet["haut"])).mean()
print(f"mode complet : MAE {mae:,.0f} EUR (artefact : {meta['metriques']['mae']:,})")
print(f"               couverture {couverture:.2%} (artefact : {meta['couverture_test']:.2%})")
assert round(mae) == meta["metriques"]["mae"], "reproduction des splits fausse — STOP"
assert round(couverture, 4) == meta["couverture_test"], "reproduction des splits fausse — STOP"
print("reproduction exacte : les mesures qui suivent decrivent bien le modele servi.")


mode complet : MAE 1,448 EUR (artefact : 1,448)
               couverture 79.48% (artefact : 79.48%)
reproduction exacte : les mesures qui suivent decrivent bien le modele servi.


## 2. Expérience A — annonce complète vs « formulaire minimal »

Les mêmes 3 977 annonces, mais avec les sept champs de l'accordéon rendus manquants — comme
si chaque vendeur n'avait rempli que la première section du formulaire. Tout le reste
(marque, modèle, année, kilométrage, énergie, boîte, état) est conservé.

Trois questions :

- de **combien le central bouge-t-il**, et dans quel sens ? (le constat utilisateur) ;
- la **largeur s'élargit-elle** ? — c'est ce que l'adaptativité devrait faire quand
  l'information manque ; si le centre plonge à largeur constante, c'est un biais, pas de
  l'incertitude ;
- la **garantie de couverture** (79,5 % mesurée en mode complet) tient-elle encore ?


In [4]:
ACCORDEON_NUM = ["puissance_din", "puissance_fisc", "critair", "portes", "places",
                 "ct_valide_jusqu_a"]

df_min = df_test.copy()
df_min[ACCORDEON_NUM] = np.nan
df_min["couleur"] = pd.NA          # construire_jeu la rabat sur "(inconnu)"

minimal = fourchette(df_min)

delta = minimal["central"] - complet["central"]
print("--- deplacement de l'estimation centrale (minimal - complet), en EUR ---")
print(delta.describe().round(0).to_string())
print()
print(f"part des annonces dont l'estimation BAISSE : {(delta < 0).mean():.1%}")
print(f"deplacement median : {delta.median():+,.0f} EUR "
      f"({100 * delta.median() / y_test.median():+.1f} % du prix median)")


--- deplacement de l'estimation centrale (minimal - complet), en EUR ---
count     3977.0
mean     -1425.0
std       3277.0
min     -30406.0
25%      -1780.0
50%       -506.0
75%        186.0
max       5475.0

part des annonces dont l'estimation BAISSE : 68.5%
deplacement median : -506 EUR (-12.1 % du prix median)


In [5]:
tranches = pd.cut(y_test, TRANCHES)
g = lambda s: s.groupby(tranches, observed=True)

bilan = pd.DataFrame({
    "n": g(y_test).size(),
    "prix_median": g(y_test).median().round(0),
    "delta_central_median": g(delta).median().round(0),
    "largeur_complet": g(complet["largeur"]).median().round(0),
    "largeur_minimal": g(minimal["largeur"]).median().round(0),
})
bilan["delta_pct_prix"] = (100 * bilan["delta_central_median"] / bilan["prix_median"]).round(1)
print(bilan.to_string())


                   n  prix_median  delta_central_median  largeur_complet  largeur_minimal  delta_pct_prix
prix_eur                                                                                                 
(500, 2000]     1061       1200.0                -169.0           1914.0           3027.0           -14.1
(2000, 5000]    1101       3400.0                -188.0           2874.0           4787.0            -5.5
(5000, 10000]    831       7000.0                -619.0           4214.0           6714.0            -8.8
(10000, 20000]   657      14500.0               -3406.0           6072.0           9489.0           -23.5
(20000, 50000]   250      27000.0               -9260.0          13744.0          14385.0           -34.3


In [6]:
mae_min = mean_absolute_error(y_test, minimal["central"])
couv_min = ((y_test >= minimal["bas"]) & (y_test <= minimal["haut"])).mean()

print("---            complet    minimal ---")
print(f"MAE            {mae:>7,.0f}    {mae_min:>7,.0f} EUR")
print(f"couverture     {couverture:>7.2%}    {couv_min:>7.2%}   (cible 80 %)")
print(f"largeur med.   {complet['largeur'].median():>7,.0f}    {minimal['largeur'].median():>7,.0f} EUR")


---            complet    minimal ---
MAE              1,448      2,544 EUR
couverture      79.48%     81.97%   (cible 80 %)
largeur med.     3,399      5,493 EUR


**Lecture.** Le constat utilisateur est confirmé, et précisé :

- **Le central est tiré vers le bas** : déplacement médian de **−506 €** (−12 % du prix
  médian), 68,5 % des annonces baissent. L'effet **croît avec le prix** : −169 € en entrée de
  gamme, **−3 406 €** entre 10 et 20 k€, **−9 260 €** (−34 %) au-delà de 20 k€. Plus la
  voiture est chère, plus « champs vides » la fait ressembler aux annonces bâclées — qui sont
  des voitures bon marché.
- **La fourchette, elle, reste honnête** : la largeur médiane passe de 3 399 € à **5 493 €**
  (+62 %) — l'adaptativité joue son rôle — et la couverture **tient** (81,97 %, au-dessus de
  la cible). Les bornes disent vrai ; c'est l'**estimation centrale** qui est biaisée
  (MAE 1 448 € → 2 544 €).

Le défaut n'est donc pas « le service ment » mais « le chiffre le plus visible de la page
(le central) est le moins fiable quand l'information manque ».

## 3. Expérience B — d'où vient le réflexe « manquant = pas cher » ?

Si l'hypothèse MNAR est vraie, elle doit se voir **dans les données** avant même le modèle :
les annonces qui omettent un champ devraient afficher des prix nettement plus bas que celles
qui le renseignent. Prix médian avec / sans, champ par champ, sur le jeu nettoyé complet.


In [7]:
CHAMPS_ACCORDEON = ACCORDEON_NUM + ["couleur"]

lignes = []
for c in CHAMPS_ACCORDEON:
    present = df[c].notna()
    lignes.append({
        "champ": c,
        "part_renseignee": round(present.mean(), 3),
        "prix_median_renseigne": round(df.loc[present, "prix_eur"].median()),
        "prix_median_manquant": (round(df.loc[~present, "prix_eur"].median())
                                  if (~present).any() else None),
    })
mnar = pd.DataFrame(lignes).set_index("champ")
mnar["ecart"] = mnar["prix_median_manquant"] - mnar["prix_median_renseigne"]
print(mnar.to_string())


                   part_renseignee  prix_median_renseigne  prix_median_manquant  ecart
champ                                                                                 
puissance_din                0.979                   4450                  3300  -1150
puissance_fisc               0.984                   4400                  4275   -125
critair                      0.331                   6000                  3600  -2400
portes                       0.996                   4400                  3500   -900
places                       0.996                   4400                  3500   -900
ct_valide_jusqu_a            0.537                   4620                  4000   -620
couleur                      0.996                   4400                  3150  -1250


## 4. Expérience C — le cas utilisateur, rejoué par le service

La Clio de référence des fiches de décision (RENAULT Clio 2015, 120 000 km, Diesel, boîte
manuelle, usure normale), passée dans `app/src/prix.estimer()` — le vrai code du service,
pas une reconstruction — accordéon vide puis rempli avec des valeurs plausibles pour ce
véhicule (90 ch DIN, 5 CV fiscaux, Crit'Air 2, 5 portes, 5 places, CT 2026, couleur grise).


In [8]:
import prix as service

clio = {"marque": "RENAULT", "modele": "Clio", "annee_mec": 2015, "kilometrage": 120_000,
        "energie": "Diesel", "boite": "Manuelle", "etat": "3_usure"}

gris = next((c for c in service.COULEURS if "gris" in c.lower()), service.COULEURS[0])
complements = {"puissance_din": 90, "puissance_fisc": 5, "critair": 2, "portes": 5,
               "places": 5, "ct_valide_jusqu_a": 2026, "couleur": gris}

r_vide = service.estimer(clio)
r_plein = service.estimer(clio | complements)

def _aff(nom, r):
    print(f"{nom:<18} {r['bas']:>7,} — {r['haut']:>7,} EUR   "
          f"central {r['central']:>7,} EUR   largeur {r['largeur']:>6,}")

_aff("accordeon vide", r_vide)
_aff("accordeon rempli", r_plein)
print(f"\necart sur le central : {r_plein['central'] - r_vide['central']:+,} EUR")


accordeon vide       4,716 —  10,195 EUR   central   7,479 EUR   largeur  5,478
accordeon rempli     5,850 —   8,592 EUR   central   8,066 EUR   largeur  2,742

ecart sur le central : +587 EUR


## Conclusion

1. **Constat confirmé et chiffré.** Accordéon vide : le central baisse pour 68,5 % des
   annonces (médiane −506 €), jusqu'à **−34 %** au-dessus de 20 000 €. Sur la Clio de
   référence, +587 € une fois l'accordéon rempli, et une fourchette deux fois plus étroite
   (5 478 € → 2 742 €).
2. **Hypothèse MNAR vérifiée dans les données** : les annonces sans Crit'Air affichent un
   prix médian de 3 600 € contre 6 000 € avec (−2 400 €) ; sans couleur −1 250 €, sans
   puissance DIN −1 150 €. Le modèle n'a rien inventé, il a appris ce que les données
   disaient — mais ce signal ne veut rien dire pour un vendeur qui ne connaît pas sa
   puissance DIN.
3. **La garantie de couverture n'est pas menacée** : 81,97 % en mode minimal, la fourchette
   s'élargit comme elle le doit. Seul le central est biaisé.

**Remèdes candidats pour la marche suivante** (décision à prendre, ADR à l'appui) :

- **Interface** (le moins risqué) : quand l'accordéon est vide, afficher la fourchette en
  premier et le central en retrait, avec un message chiffré « estimation moins précise —
  renseigner ces champs la resserre » ;
- **Imputation au service** (médiane par marque/modèle) : recentre le central mais réintroduit
  un décalage entraînement/service — à mesurer avant d'adopter ;
- **Ré-entraînement avec masquage aléatoire** (augmentation de manquance) : casse la
  corrélation « manquant = pas cher » à la source, mais coûte un ré-entraînement et une
  re-calibration.